In [1]:
import os

date = "251128"
dl_folder = f"D:/Python_TK_3/datas/{date}_DL"

dl_number = "Vis_angle_multi_09"
os.mkdir(f"{dl_folder}/model_{dl_number}")

In [2]:
experiments_list1 = [
#                    "250829_01",
                    "250903_02",
                    "250905_01",
                    "250909_02",
                    "250911_01",
                    "250916_02",
                    "250918_02",
                    "250922_02",
                    "250924_02",
                    "250929_02",
                    "251001_02",
                    "251007_02",
                    "251009_02",
                    ]
mouse_id1 = "250328TK_WF_ECGCP"

experiments_list2 = [
                    "250901_02",
                    "250903_05",
                    "250905_03",
                    "250909_04",
                    "250911_03",
#                    "250916_03",
                    "250918_04",
                    "250922_04",
                    "250924_04",
                    "250926_04",
#                    "250929_04",
                    "251001_04",
                    "251003_04",
                    "251007_04",
                    "251009_04",
                    ]
mouse_id2 = "250807TK_WF_ECGCP"

experiments_list3 = [
                    "250930_02",
                    "251002_02",
                    "251006_02",
                    "251010_02",
                    "251014_02",
                    "251016_02",
                    "251020_02",
                    "251022_02",
                    "251024_02",
                    "251027_02",
                    "251029_02",
                    "251031_02",
                    ]
mouse_id3 = "250902TK_WF_ECGCP"


ex_list = [
            experiments_list1,
            experiments_list2, 
            experiments_list3,  
                ]

mouse_list = [
              mouse_id1,
              mouse_id2,
              mouse_id3,  
                ]

In [3]:
fs = 20  # サンプリング周波数
calc_start = 5
calc_end = 39
# データのサンプリングレート
first_ex = 0
last_ex = 174
#last_ex = 58
NumberOfDatas = last_ex - first_ex + 1        # number of experiments
start_stim = 20
stop_stim = 30
start_ave= 10
end_ave= 20
look_frame = 30     

In [4]:
import numpy as np
from scipy.signal import butter, filtfilt

def lowpass_filter(data, cutoff):

    # ローパスフィルターの設計
    #cutoff = cutoff  # カットオフ周波数 (Hz)
    order = 4  # フィルターの次数
    nyquist = 0.5 * fs  # ナイキスト周波数
    normal_cutoff = cutoff / nyquist  # 正規化カットオフ周波数

    # Butterworthフィルターの設計
    b, a = butter(order, normal_cutoff, btype='low', analog=False)

    # フィルターの適用
    filtered_data = filtfilt(b, a, data)

    return filtered_data

In [5]:
import numpy as np

def normalize(data, start_n, end_n):

    norm_data = data / np.mean(data[start_n:end_n])
    
    return norm_data

In [6]:
import numpy as np

def calc_velocity(data, dt):
     
    velocity = []
    for frame in range(data.shape[0]-1):
        vel_frame = (data[frame+1]-data[frame]) / dt
        velocity.append(vel_frame)
    velocity = np.array(velocity)

    return velocity

In [7]:
import numpy as np

def z_score(data, start_n, end_n):

    z_data = (data-np.mean(data[start_n:end_n])) / np.std(data[start_n:end_n])
    
    return z_data

In [8]:
import numpy as np

def z_score2(data, start_n, end_n):

    z_data = (data-np.mean(data[start_n:end_n])) / np.std(data[start_n:end_n])
    
    return z_data, np.mean(data[start_n:end_n]), np.std(data[start_n:end_n])

In [9]:
import numpy as np

def z_score3(data, data_base, start_n, end_n):

    z_data = (data-np.mean(data_base[start_n:end_n])) / np.std(data_base[start_n:end_n])
    
    return z_data, np.mean(data_base[start_n:end_n]), np.std(data_base[start_n:end_n])

In [10]:
import numpy as np

def min_max_norm(data):

    min_max = (data-min(data)) / (max(data)-min(data))
    
    return min_max

In [11]:
import numpy as np

def min_max_norm2(data):

    min_max = (data-min(data)) / (max(data)-min(data))
    
    return min_max, min(data), max(data)

In [12]:
import numpy as np

def calc_response(data, base_start, base_end):
    baseline = np.mean(data[base_start:base_end], axis=0)
    response = data - baseline

    return response

In [13]:
import numpy as np
import tqdm

no_cycle = 10
pupil_cutoff = 0.2
vis_stim1 = 15
vis_stim2 = 25
use_range = 5

diameters = []
velocities = []
accelerations = []
for mouse in tqdm.tqdm(range(len(mouse_list))):
    diameters1, diameters2 = [], []
    velocities1, velocities2 = [], []
    accelerations1, accelerations2 = [], []
    for ex in range(len(ex_list[mouse])):
        diameter1, diameter2 = [], []
        velocity1, velocity2 = [], []
        acceleration1, acceleration2 = [], []
        for cycle in range(no_cycle):
            diameter = np.load(f"D:/Python_TK_3/datas/pupil_data/dlc_model_02/{mouse_list[mouse]}/{ex_list[mouse][ex]}/{ex_list[mouse][ex]}_ex0{cycle}.npy")
            diameter = lowpass_filter(data=diameter, cutoff=pupil_cutoff)
            n_diameter = normalize(data=diameter, start_n=20, end_n=90)
            velocity = calc_velocity(data=n_diameter, dt=1/fs)
            acceleration = calc_velocity(data=velocity, dt=1/fs)

            temp_dia1 = n_diameter[int(fs*vis_stim1)-int(fs*use_range):int(fs*vis_stim1)+int(fs*use_range)]
            temp_dia2 = n_diameter[int(fs*vis_stim2)-int(fs*use_range):int(fs*vis_stim2)+int(fs*use_range)]
            temp_velo1 = velocity[int(fs*vis_stim1)-int(fs*use_range)-1:int(fs*vis_stim1)+int(fs*use_range)-1]
            temp_velo2 = velocity[int(fs*vis_stim2)-int(fs*use_range)-1:int(fs*vis_stim2)+int(fs*use_range)-1]
            temp_acc1 = acceleration[int(fs*vis_stim1)-int(fs*use_range)-2:int(fs*vis_stim1)+int(fs*use_range)-2]
            temp_acc2 = acceleration[int(fs*vis_stim2)-int(fs*use_range)-2:int(fs*vis_stim2)+int(fs*use_range)-2]

            diameter1.append(temp_dia1)
            diameter2.append(temp_dia2)
            velocity1.append(temp_velo1)
            velocity2.append(temp_velo2)
            acceleration1.append(temp_acc1)
            acceleration2.append(temp_acc2)


        diameters1.append(np.array(diameter1))
        diameters2.append(np.array(diameter2))
        velocities1.append(np.array(velocity1))
        velocities2.append(np.array(velocity2))
        accelerations1.append(np.array(acceleration1))
        accelerations2.append(np.array(acceleration2))

    diameters.append([diameters1, diameters2])
    velocities.append([velocities1, velocities2])
    accelerations.append([accelerations1, accelerations2])

#diameters = np.array(diameters)
#velocities = np.array(velocities)

print(len(diameters), len(diameters[0]), len(diameters[0][0]), diameters[0][0][0].shape)
print(len(velocities), len(velocities[0]), len(velocities[0][0]), velocities[0][0][0].shape)
print(len(accelerations), len(accelerations[0]), len(accelerations[0][0]), accelerations[0][0][0].shape)
print(len(diameters), len(diameters[1]), len(diameters[1][0]), diameters[1][0][0].shape)
print(len(velocities), len(velocities[1]), len(velocities[1][0]), velocities[1][0][0].shape)
print(len(accelerations), len(accelerations[1]), len(accelerations[1][0]), accelerations[1][0][0].shape)
print(len(diameters), len(diameters[2]), len(diameters[2][0]), diameters[2][0][0].shape)
print(len(velocities), len(velocities[2]), len(velocities[2][0]), velocities[2][0][0].shape)
print(len(accelerations), len(accelerations[2]), len(accelerations[2][0]), accelerations[2][0][0].shape)

100%|██████████| 3/3 [00:04<00:00,  1.44s/it]

3 2 12 (10, 200)
3 2 12 (10, 200)
3 2 12 (10, 200)
3 2 13 (10, 200)
3 2 13 (10, 200)
3 2 13 (10, 200)
3 2 12 (10, 200)
3 2 12 (10, 200)
3 2 12 (10, 200)


In [14]:
dia_list, vel_list, acc_list = [], [], []
for mouse in tqdm.tqdm(range(len(mouse_list))):
    dia_array = np.array(diameters[mouse]).reshape(len(diameters[mouse])*len(diameters[mouse][0])*diameters[mouse][0][0].shape[0], diameters[mouse][0][0].shape[1])
    vel_array = np.array(velocities[mouse]).reshape(len(velocities[mouse])*len(velocities[mouse][0])*velocities[mouse][0][0].shape[0], velocities[mouse][0][0].shape[1])
    acc_array = np.array(accelerations[mouse]).reshape(len(accelerations[mouse])*len(accelerations[mouse][0])*accelerations[mouse][0][0].shape[0], accelerations[mouse][0][0].shape[1])

    dia_list.append(dia_array)
    vel_list.append(vel_array)
    acc_list.append(acc_array)

print(len(dia_list), len(vel_list), len(acc_list))
print(dia_list[0].shape, vel_list[0].shape, acc_list[0].shape)
print(dia_list[1].shape, vel_list[1].shape, acc_list[1].shape)
print(dia_list[2].shape, vel_list[2].shape, acc_list[2].shape)

100%|██████████| 3/3 [00:00<?, ?it/s]

3 3 3
(240, 200) (240, 200) (240, 200)
(260, 200) (260, 200) (260, 200)
(240, 200) (240, 200) (240, 200)


In [15]:
base_start, base_end = 85, 95

dia_res, vel_res, acc_res = [], [], []
for mouse in tqdm.tqdm(range(len(mouse_list))):
    temp_dia, temp_vel, temp_acc = [], [], []
    for ex in range(dia_list[mouse].shape[0]):
        temp_dia.append(calc_response(dia_list[mouse][ex], base_start, base_end))
        temp_vel.append(calc_response(vel_list[mouse][ex], base_start, base_end))
        temp_acc.append(calc_response(acc_list[mouse][ex], base_start, base_end))
    dia_res.append(np.array(temp_dia))
    vel_res.append(np.array(temp_vel))
    acc_res.append(np.array(temp_acc))

print(len(dia_res), len(vel_res), len(acc_res))
print(dia_res[0].shape, vel_res[0].shape, acc_res[0].shape)
print(dia_res[1].shape, vel_res[1].shape, acc_res[1].shape)
print(dia_res[2].shape, vel_res[2].shape, acc_res[2].shape)

100%|██████████| 3/3 [00:00<00:00, 137.04it/s]

3 3 3
(240, 200) (240, 200) (240, 200)
(260, 200) (260, 200) (260, 200)
(240, 200) (240, 200) (240, 200)


In [16]:
start_n, end_n = 0, 200

dia_z, vel_z, acc_z = [], [], []
for mouse in tqdm.tqdm(range(len(mouse_list))):
    temp_dia, temp_vel, temp_acc = [], [], []
    for ex in range(dia_res[mouse].shape[0]):
        temp_dia.append(z_score(dia_res[mouse][ex], start_n, end_n))
        temp_vel.append(z_score(vel_res[mouse][ex], start_n, end_n))
        temp_acc.append(z_score(acc_res[mouse][ex], start_n, end_n))
    dia_z.append(np.array(temp_dia))
    vel_z.append(np.array(temp_vel))
    acc_z.append(np.array(temp_acc))

print(len(dia_z), len(vel_z), len(acc_z))
print(dia_z[0].shape, vel_z[0].shape, acc_z[0].shape)
print(dia_z[1].shape, vel_z[1].shape, acc_z[1].shape)
print(dia_z[2].shape, vel_z[2].shape, acc_z[2].shape)

100%|██████████| 3/3 [00:00<00:00, 31.99it/s]

3 3 3
(240, 200) (240, 200) (240, 200)
(260, 200) (260, 200) (260, 200)
(240, 200) (240, 200) (240, 200)


In [17]:
dia_mm, vel_mm, acc_mm = [], [], []
for mouse in tqdm.tqdm(range(len(mouse_list))):
    temp_dia, temp_vel, temp_acc = [], [], []
    for ex in range(dia_res[mouse].shape[0]):
        temp_dia.append(min_max_norm(dia_z[mouse][ex]))
        temp_vel.append(min_max_norm(vel_z[mouse][ex]))
        temp_acc.append(min_max_norm(acc_z[mouse][ex]))
    dia_mm.append(np.array(temp_dia))
    vel_mm.append(np.array(temp_vel))
    acc_mm.append(np.array(temp_acc))

print(len(dia_mm), len(vel_mm), len(acc_mm))
print(dia_mm[0].shape, vel_mm[0].shape, acc_mm[0].shape)
print(dia_mm[1].shape, vel_mm[1].shape, acc_mm[1].shape)
print(dia_mm[2].shape, vel_mm[2].shape, acc_mm[2].shape)

100%|██████████| 3/3 [00:00<00:00, 20.61it/s]

3 3 3
(240, 200) (240, 200) (240, 200)
(260, 200) (260, 200) (260, 200)
(240, 200) (240, 200) (240, 200)


In [18]:
refer_pupil = dia_mm

In [19]:
import numpy as np
import tqdm

hemo_folder = "D:/Python_TK_3/datas/hemo_corrected"
cortex_cutoff = 0.1

ca_datas = []
for mouse in tqdm.tqdm(range(len(mouse_list))):
    img_path = f"{hemo_folder}/img/{mouse_list[mouse]}"
    roi_path = f"{hemo_folder}/roi/{mouse_list[mouse]}"

    ca_data = []
    for ex in range(len(ex_list[mouse])):
        ca_data1, ca_data2 = [], []
        for cycle in range(no_cycle):
            #cycle_ca =  np.load(f"{img_path}/{ex_list[ex]}/{ex_list[ex]}_ex0{cycle}.npy")
            cycle_ca = np.load(f"{roi_path}/{ex_list[mouse][ex]}/{ex_list[mouse][ex]}_ex0{cycle}.npy")
            #cycle_ca = lowpass_filter(data=cycle_ca, cutoff=cortex_cutoff)

            z_ca = []
            for area in range(cycle_ca.shape[0]):
                low_ca = lowpass_filter(data=cycle_ca[area], cutoff=cortex_cutoff)
                temp_z = z_score(data=low_ca, start_n=20, end_n=90)
                z_ca.append(np.array(temp_z))
            cycle_ca1 = np.array(z_ca)[:, int(fs*vis_stim1)-int(fs*use_range):int(fs*vis_stim1)+int(fs*use_range)]
            cycle_ca2 = np.array(z_ca)[:, int(fs*vis_stim2)-int(fs*use_range):int(fs*vis_stim2)+int(fs*use_range)]

            ca_data1.append(cycle_ca1)
            ca_data2.append(cycle_ca2)

        ca_data1, ca_data2 = np.array(ca_data1), np.array(ca_data2)
        ca_data_n = np.concatenate((ca_data1, ca_data2), axis=0)
        #ca_data_n = np.concatenate((ca_data1.transpose(1, 0, 2), ca_data2.transpose(1, 0, 2)), axis=0)

        ca_data.append(ca_data_n)
    #ca_datas.append(ca_data)
    ca_datas.append(np.array(ca_data))

print(ca_datas[0].shape)
print(ca_datas[1].shape)
print(ca_datas[2].shape)

100%|██████████| 3/3 [00:07<00:00,  2.46s/it]

(12, 20, 26, 200)
(13, 20, 26, 200)
(12, 20, 26, 200)


In [20]:
ca_list = []
for mouse in tqdm.tqdm(range(len(mouse_list))):
    temp_ca = []
    for ex in range(len(ca_datas[mouse])):
        if ex == 0:
            ca_array = ca_datas[mouse][ex]
        else:
            ca_array = np.concatenate((ca_array, ca_datas[mouse][ex]), axis=0)
    ca_list.append(ca_array)
    

print(ca_list[0].shape, ca_list[1].shape, ca_list[2].shape)

100%|██████████| 3/3 [00:00<00:00, 48.78it/s]

(240, 26, 200) (260, 26, 200) (240, 26, 200)


In [21]:
min_max_ca = []
for mouse in tqdm.tqdm(range(len(mouse_list))):
    temp_ca = []
    for ex in range(ca_list[mouse].shape[0]):
        area_ca = []
        for area in range(ca_list[mouse].shape[1]):
            temp_mm = min_max_norm(ca_list[mouse][ex, area])
            area_ca.append(np.array(temp_mm))
        temp_ca.append(area_ca)
    min_max_ca.append(np.array(temp_ca))

ca_list = min_max_ca

print(ca_list[0].shape, ca_list[1].shape, ca_list[2].shape)

100%|██████████| 3/3 [00:01<00:00,  1.98it/s]

(240, 26, 200) (260, 26, 200) (240, 26, 200)


In [22]:
ca_list2 = np.load("D:/Python_TK_3/datas/251110_ica/250829-251031_vis_angle_v8/ic_temporal_components_extracted.npy")
#ca_list2 = np.load("D:/Python_TK_3/datas/251110_ica/250829-251031_vis_angle_v14/ic_temporal_components_extracted.npy")
print(ca_list2.shape)  #(models, experiments, components, frames)

(23, 740, 200)


In [23]:
ca_list2 = ca_list2.transpose(1, 0, 2)
print(ca_list2.shape)

(740, 23, 200)


In [24]:
start_n, end_n = 0, 50
feature_cutoff = 2

means, stds = np.zeros_like(ca_list2[:,:,0]), np.zeros_like(ca_list2[:,:,0])
mins, maxs  = np.zeros_like(ca_list2[:,:,0]), np.zeros_like(ca_list2[:,:,0])
for ex in range(ca_list2.shape[0]):
    for comps in range(ca_list2.shape[1]):
        ca_list2[ex, comps] = lowpass_filter(ca_list2[ex, comps], cutoff=feature_cutoff)
        #ca_list2[mouse, ex, comps] = z_score(ca_list2[mouse, ex, comps], start_n, end_n)
        if ex % 2 == 0:
            ca_list2[ex, comps], means[ex, comps], stds[ex, comps] = z_score2(ca_list2[ex, comps], start_n, end_n)
        else:
            ca_list2[ex, comps], means[ex, comps], stds[ex, comps] = z_score3(ca_list2[ex, comps], ca_list2[ex-1, comps], start_n, end_n)
        ca_list2[ex, comps], mins[ex, comps], maxs[ex, comps] = min_max_norm2(ca_list2[ex, comps])

print(ca_list2.shape)  #(mice, ,components, frames)
print(means.shape, stds.shape, mins.shape, maxs.shape)

(740, 23, 200)
(740, 23) (740, 23) (740, 23) (740, 23)


In [25]:
means = [means[:ca_list[0].shape[0]], means[ca_list[0].shape[0]:ca_list[0].shape[0]+ca_list[1].shape[0]], means[ca_list[0].shape[0]+ca_list[1].shape[0]:ca_list[0].shape[0]+ca_list[1].shape[0]+ca_list[2].shape[0]]]
print(len(means), means[0].shape, means[1].shape, means[2].shape)

stds = [stds[:ca_list[0].shape[0]], stds[ca_list[0].shape[0]:ca_list[0].shape[0]+ca_list[1].shape[0]], stds[ca_list[0].shape[0]+ca_list[1].shape[0]:ca_list[0].shape[0]+ca_list[1].shape[0]+ca_list[2].shape[0]]]
print(len(stds), stds[0].shape, stds[1].shape, stds[2].shape)

mins = [mins[:ca_list[0].shape[0]], mins[ca_list[0].shape[0]:ca_list[0].shape[0]+ca_list[1].shape[0]], mins[ca_list[0].shape[0]+ca_list[1].shape[0]:ca_list[0].shape[0]+ca_list[1].shape[0]+ca_list[2].shape[0]]]
print(len(mins), mins[0].shape, mins[1].shape, mins[2].shape)

maxs = [maxs[:ca_list[0].shape[0]], maxs[ca_list[0].shape[0]:ca_list[0].shape[0]+ca_list[1].shape[0]], maxs[ca_list[0].shape[0]+ca_list[1].shape[0]:ca_list[0].shape[0]+ca_list[1].shape[0]+ca_list[2].shape[0]]]
print(len(maxs), maxs[0].shape, maxs[1].shape, maxs[2].shape)

3 (240, 23) (260, 23) (240, 23)
3 (240, 23) (260, 23) (240, 23)
3 (240, 23) (260, 23) (240, 23)
3 (240, 23) (260, 23) (240, 23)


In [26]:
use_cal = [ca_list2[:ca_list[0].shape[0]], ca_list2[ca_list[0].shape[0] : ca_list[0].shape[0]+ca_list[1].shape[0]], ca_list2[ca_list[0].shape[0]+ca_list[1].shape[0] : ca_list[0].shape[0]+ca_list[1].shape[0]+ca_list[2].shape[0]]]

print(len(use_cal))
print(use_cal[0].shape, use_cal[1].shape, use_cal[2].shape)

3
(240, 23, 200) (260, 23, 200) (240, 23, 200)


In [27]:
region_num = 2
# window = int(fs*0.2)
back_second = 2.0
forward_second = 0.5
stim_frame = 100
start_frame = 0
end_frame = 80
pupil_thresh = 0.6

pupil_list = []
cortex_list = []
index_list =[]
area_list = []
corr_array = []
for mouse in tqdm.tqdm(range(len(mouse_list))):
    binary_pupil = []
    extracted_signal = []
    extracted_z_area = []
    max_idxs = []
    for ex in range(refer_pupil[mouse].shape[0]):
        max_idx = np.argmax(refer_pupil[mouse][ex, stim_frame+10:-10]) + stim_frame+10
        #max_idx = np.argmax(dia_mm[mouse][ex, stim_frame+4:-2]) + stim_frame+4
        #max_idx = np.argmax(dia_list[mouse][ex, stim_frame:-10]) + stim_frame
        #extracted_pupil = vel_list[mouse][ex, max_idx-int(fs*back_second)-look_frame:max_idx+int(fs*forward_second)]
        extracted_pupil = refer_pupil[mouse][ex, max_idx-int(fs*back_second)-look_frame:max_idx+int(fs*forward_second)]
        #z_calcium = []
        #for area in range(ca_list[mouse].shape[1]):
        #    temp_z = z_score_calc(ca_list[mouse][ex, area], start_frame, end_frame)
        #    z_calcium.append(temp_z)
        z_calcium = ca_list[mouse][ex]
        #use_calcium = np.stack((use_cal[mouse][ex], use_cal[mouse][ex], use_cal[mouse][ex]), axis=0)
        use_calcium = use_cal[mouse][ex]
        #extracted_ca = ca_list[mouse][ex, :, max_idx-int(fs*back_second)-look_frame:max_idx+int(fs*forward_second)]
        #extracted_area = z_calcium[:, max_idx-int(fs*back_second)-look_frame:max_idx+int(fs*forward_second)]
        extracted_area = use_calcium[:, max_idx-int(fs*back_second)-look_frame:max_idx+int(fs*forward_second)]
        extracted_ca   = use_calcium[:, max_idx-int(fs*back_second)-look_frame:max_idx+int(fs*forward_second)]
        #roi_z = z_score_calc(ca_array[ex, region_num])
        #roi_z = extracted_ca[region_num]
        #extracted_z = signal_z[ex, region_num, max_idx-int(fs*0.65)-look_frame-window+2:max_idx+int(fs*0.05)-window+2]
        #extracted_z = roi_z[max_idx-int(fs*back_second)-look_frame:max_idx+int(fs*forward_second)]

        temp_corr = []
        for comp in range(extracted_ca.shape[0]):
            #temp_corr.append(np.corrcoef(extracted_pupil[look_frame+int(fs*back_second):], extracted_ca[comp, look_frame+int(fs*back_second):])[0, 1])
            temp_corr.append(np.corrcoef(extracted_pupil[look_frame:], extracted_ca[comp, look_frame:])[0, 1])

        temp_p = []
        for f in range(extracted_pupil.shape[0]):
            if extracted_pupil[f] > pupil_thresh:
                temp_p.append(1)
            else:
                temp_p.append(0)

        binary_pupil.append(temp_p)
        max_idxs.append(max_idx)
        #extracted_signal.append(extracted_ca.transpose(0, 2, 1))
        extracted_signal.append(extracted_ca.transpose())
        extracted_z_area.append(extracted_area.transpose())
        corr_array.append(temp_corr)

    binary_pupil = np.array(binary_pupil)
    extracted_signal = np.array(extracted_signal)
    extracted_z_area = np.array(extracted_z_area)

    pupil_list.append(binary_pupil)
    cortex_list.append(extracted_signal)
    index_list.append(max_idxs)
    area_list.append(extracted_z_area)

corr_array = np.array(corr_array)

print(pupil_list[0].shape, cortex_list[0].shape, area_list[0].shape)
print(pupil_list[1].shape, cortex_list[1].shape, area_list[1].shape)
print(pupil_list[2].shape, cortex_list[2].shape, area_list[2].shape)
print(corr_array.shape)

100%|██████████| 3/3 [00:01<00:00,  2.15it/s]

(240, 80) (240, 80, 23) (240, 80, 23)
(260, 80) (260, 80, 23) (260, 80, 23)
(240, 80) (240, 80, 23) (240, 80, 23)
(740, 23)


In [28]:
mean_corr = np.mean(corr_array, axis=0)
print(mean_corr.shape)
print(np.argmax(mean_corr))
print(mean_corr[np.argmax(mean_corr)])

(23,)
20
0.045529800344382546


In [29]:
Pupil_data = pupil_list
signal = cortex_list

peak_frame = int(fs*back_second)
threshold = 0.95
area = np.argmax(mean_corr)
cor_thresh = 0.25

target = []
feature = []
experiment_list = []
mean_array, std_array = [], []
min_array, max_array = [], []

pupil_peaks = []
cortex_peaks = []

for mouse in tqdm.tqdm(range(len(mouse_list))):
    temp_target = []
    temp_feature = []
    temp_experiment_list = []
    temp_pupil, temp_cortex = [], []
    temp_mean, temp_std = [], []
    temp_min, temp_max = [], []
    for ex in range(Pupil_data[mouse].shape[0]):
        peak_pupil = max(refer_pupil[mouse][ex])
        peak_cortex = max(area_list[mouse][ex, :, area])
        temp_pupil.append(peak_pupil)
        temp_cortex.append(peak_cortex)
        #if np.all(Pupil_data[mouse][ex, :int(fs*back_second)] == 0) and np.any(Pupil_data[mouse][ex, int(fs*back_second):] == 1) and np.any(signal[mouse][ex, int(fs*back_second):, area] > threshold):
        #if np.any(Pupil_data[mouse][ex, int(fs*back_second):] == 1) and np.any(signal[mouse][ex, int(fs*back_second):, area] > threshold):
        #if np.any(signal[mouse][ex, int(fs*back_second):, area] > threshold):
        #if np.all(Pupil_data[mouse][ex, :5] == 0) and np.any(Pupil_data[mouse][ex, 5:] == 1) and np.any(signal[mouse][ex, int(fs*back_second):, area] > threshold):
        #if np.all(Pupil_data[mouse][ex, :5] == 0) and np.any(signal[mouse][ex, int(fs*back_second):, area] > threshold):
        #if np.all(Pupil_data[mouse][ex, :5] == 0) and np.any(Pupil_data[mouse][ex, 5:] == 1) and np.any(signal[mouse][ex, 5:, area] > threshold):
        #if np.any(Pupil_data[mouse][ex, :int(fs*back_second)] == 0) and np.any(Pupil_data[mouse][ex, int(fs*back_second):] == 1) and np.any(signal[mouse][ex, int(fs*back_second):, area] > threshold):
        #if np.count_nonzero(Pupil_data[mouse][ex, look_frame:] == 0) > 20 and np.count_nonzero(Pupil_data[mouse][ex, look_frame:] == 1) > 20 and np.any(signal[mouse][ex, int(fs*back_second):, area] > threshold):
        #if np.count_nonzero(Pupil_data[mouse][ex, look_frame:] == 0) > 20 and np.count_nonzero(Pupil_data[mouse][ex, look_frame:] == 1) > 20 and np.any(area_list[mouse][ex, int(fs*back_second):, area] > threshold) and np.corrcoef(Pupil_data[mouse][ex, look_frame:], area_list[mouse][ex, look_frame:, area])[0, 1]>cor_thresh:            
        #if np.count_nonzero(Pupil_data[mouse][ex, look_frame:] == 0) > 20 and np.count_nonzero(Pupil_data[mouse][ex, look_frame:] == 1) > 20 and np.any(area_list[mouse][ex, int(fs*back_second):, area] > threshold) and np.corrcoef(Pupil_data[mouse][ex, look_frame:], area_list[mouse][ex, look_frame:, area])[0, 1]>cor_thresh:   
        #if np.count_nonzero(Pupil_data[mouse][ex, look_frame:] == 0) > 20 and np.count_nonzero(Pupil_data[mouse][ex, look_frame:] == 1) > 20 and np.any(area_list[mouse][ex, int(fs*back_second):, area] > threshold):  
        #if np.count_nonzero(Pupil_data[mouse][ex, look_frame:] == 0) > 20 and np.count_nonzero(Pupil_data[mouse][ex, look_frame:] == 1) > 20 and np.any(area_list[mouse][ex, int(fs*back_second):, area] > threshold):    
        #if np.count_nonzero(Pupil_data[mouse][ex, look_frame:] == 0) > 20 and np.count_nonzero(Pupil_data[mouse][ex, look_frame:] == 1) > 20 and np.all(Pupil_data[mouse][ex, look_frame+peak_frame-25:look_frame+peak_frame-17] == 0):
        #if np.count_nonzero(Pupil_data[mouse][ex, look_frame:] == 0) > 1 and np.count_nonzero(Pupil_data[mouse][ex, look_frame:] == 1) > 10 and np.all(Pupil_data[mouse][ex, look_frame:look_frame+peak_frame-27] == 0):      
        #if np.count_nonzero(Pupil_data[mouse][ex, look_frame:] == 0) > 1 and np.count_nonzero(Pupil_data[mouse][ex, look_frame:] == 1) > 20 and np.all(Pupil_data[mouse][ex, look_frame:look_frame+peak_frame-30] == 0):         
        if np.count_nonzero(Pupil_data[mouse][ex, look_frame:] == 1) > 20 and np.all(Pupil_data[mouse][ex, look_frame:look_frame+peak_frame-30] == 0): 
            temp_target.append(Pupil_data[mouse][ex])
            temp_feature.append(signal[mouse][ex].transpose())
            temp_experiment_list.append(ex)
        temp_mean.append(means[mouse][ex])
        temp_std.append(stds[mouse][ex])
        temp_min.append(mins[mouse][ex])
        temp_max.append(maxs[mouse][ex])

    target.append(np.array(temp_target))
    feature.append(np.array(temp_feature))
    experiment_list.append(np.array(temp_experiment_list))
    mean_array.append(np.array(temp_mean))
    std_array.append(np.array(temp_std))
    min_array.append(np.array(temp_min))
    max_array.append(np.array(temp_max))

    pupil_peaks.append(temp_pupil)
    cortex_peaks.append(temp_cortex)

100%|██████████| 3/3 [00:00<00:00, 101.21it/s]


In [30]:
Pupil_data = target
signal = feature
experiment_list = experiment_list

print(Pupil_data[0].shape, signal[0].shape, experiment_list[0].shape)
print(Pupil_data[1].shape, signal[1].shape, experiment_list[1].shape)
print(Pupil_data[2].shape, signal[2].shape, experiment_list[2].shape)

(42, 80) (42, 23, 80) (42,)
(59, 80) (59, 23, 80) (59,)
(42, 80) (42, 23, 80) (42,)


In [31]:
for mouse in range(len(Pupil_data)):
    print(np.sum(Pupil_data[mouse].reshape(-1))/len(Pupil_data[mouse].reshape(-1)))

0.42083333333333334
0.4226694915254237
0.42678571428571427


In [32]:
ex_numbers0, ex_numbers1, ex_numbers2 = 240, 260, 240

ex_rate = np.array([
                    experiment_list[0].shape[0]/ex_numbers0,
                    experiment_list[1].shape[0]/ex_numbers1,
                    experiment_list[2].shape[0]/ex_numbers2,
                    ])

posi_rate = np.array([
                        np.sum(Pupil_data[0].reshape(-1))/len(Pupil_data[0].reshape(-1)),
                        np.sum(Pupil_data[1].reshape(-1))/len(Pupil_data[1].reshape(-1)),
                        np.sum(Pupil_data[2].reshape(-1))/len(Pupil_data[2].reshape(-1)),
                        ])

print(ex_rate.shape, ex_rate)
print(posi_rate.shape, posi_rate)

np.save(f"{dl_folder}/model_{dl_number}/{date}_{dl_number}_ex_rate.npy", ex_rate)
np.save(f"{dl_folder}/model_{dl_number}/{date}_{dl_number}_positive_rate.npy", posi_rate)

(3,) [0.175      0.22692308 0.175     ]
(3,) [0.42083333 0.42266949 0.42678571]


In [33]:
print(experiment_list[0])
print(experiment_list[1])
print(experiment_list[2])

[  0  14  15  18  30  34  51  54  62  68  69  78  80 104 106 115 118 121
 128 131 135 142 143 144 145 146 148 149 151 153 154 167 169 171 180 184
 204 206 213 218 219 226]
[  5  12  18  26  37  42  45  47  51  55  57  58  61  63  64  67  68  72
  78  83  90  92  96 101 106 115 117 118 120 127 136 137 139 140 145 149
 150 152 153 156 158 169 172 183 185 186 187 191 193 195 199 200 204 206
 212 243 249 252 256]
[  2   3   8  21  22  26  27  28  36  48  49  50  65  67  71  81  86 107
 110 111 118 125 128 147 150 151 152 160 162 163 169 183 185 197 200 201
 204 206 212 221 223 238]


In [34]:
#import numpy as np

#size = 12

# ランダムに11要素を抽出（重複なし）
#experiment_list[1] = np.random.choice(experiment_list[1], size=size, replace=False)
#print(experiment_list[1])

#import numpy as np

# ランダムに11要素を抽出（重複なし）
#experiment_list[2] = np.random.choice(experiment_list[2], size=size, replace=False)
#print(experiment_list[2])

In [35]:
import random


mouse = 0
random.seed(123)

# 0～35 のリストを作成してシャッフル
numbers = list(range(experiment_list[mouse].shape[0]))
random.shuffle(numbers)

# 6 個ずつスライスして変数に格納
dataset1 = numbers[0:9]
dataset2 = numbers[9:18]
dataset3 = numbers[18:26]
dataset4 = numbers[26:34]
dataset5 = numbers[34:42]

In [36]:
import itertools


vars_lists = [dataset1, dataset2, dataset3, dataset4, dataset5]

splits1 = []
# 6 つから 4 つを選ぶ
for group4 in itertools.combinations(vars_lists, 4):
    remaining = [v for v in vars_lists if v not in group4]
    # 残り 2 つから 1 つを選ぶ
        # 残った最後の 1 つを取得
    #group1 = next(v for v in group4 if v not in remaining)
    # リストとして格納：[4つリスト, 1つ目, 2つ目]
    splits1.append([
        list(group4),    # 4つのリストをリスト化
        remaining,       # 1つ目のリスト
        #group2           # 2つ目のリスト
    ])

print(f"全パターン数: {len(splits1)}")  # → 30

# 確認例
for i, pat in enumerate(splits1[:5], 1):
    four, one_a = pat
    print(f"{i:2d}:")
    print("  4つ→", four)
    print("  1つA→", one_a)
    #print("  1つB→", one_b)

全パターン数: 5
 1:
  4つ→ [[32, 36, 15, 23, 29, 31, 8, 14, 9], [27, 18, 16, 1, 34, 11, 38, 25, 20], [12, 19, 35, 13, 0, 30, 7, 22], [28, 37, 10, 4, 39, 41, 33, 21]]
  1つA→ [[24, 2, 6, 40, 26, 5, 17, 3]]
 2:
  4つ→ [[32, 36, 15, 23, 29, 31, 8, 14, 9], [27, 18, 16, 1, 34, 11, 38, 25, 20], [12, 19, 35, 13, 0, 30, 7, 22], [24, 2, 6, 40, 26, 5, 17, 3]]
  1つA→ [[28, 37, 10, 4, 39, 41, 33, 21]]
 3:
  4つ→ [[32, 36, 15, 23, 29, 31, 8, 14, 9], [27, 18, 16, 1, 34, 11, 38, 25, 20], [28, 37, 10, 4, 39, 41, 33, 21], [24, 2, 6, 40, 26, 5, 17, 3]]
  1つA→ [[12, 19, 35, 13, 0, 30, 7, 22]]
 4:
  4つ→ [[32, 36, 15, 23, 29, 31, 8, 14, 9], [12, 19, 35, 13, 0, 30, 7, 22], [28, 37, 10, 4, 39, 41, 33, 21], [24, 2, 6, 40, 26, 5, 17, 3]]
  1つA→ [[27, 18, 16, 1, 34, 11, 38, 25, 20]]
 5:
  4つ→ [[27, 18, 16, 1, 34, 11, 38, 25, 20], [12, 19, 35, 13, 0, 30, 7, 22], [28, 37, 10, 4, 39, 41, 33, 21], [24, 2, 6, 40, 26, 5, 17, 3]]
  1つA→ [[32, 36, 15, 23, 29, 31, 8, 14, 9]]


In [37]:
import random


mouse = 1
random.seed(60)

# 0～35 のリストを作成してシャッフル
numbers = list(range(experiment_list[mouse].shape[0]))
random.shuffle(numbers)

# 6 個ずつスライスして変数に格納
dataset1 = numbers[0:12]
dataset2 = numbers[12:24]
dataset3 = numbers[24:36]
dataset4 = numbers[36:48]
dataset5 = numbers[48:59]

In [38]:
import itertools


vars_lists = [dataset1, dataset2, dataset3, dataset4, dataset5]

splits2 = []
# 6 つから 4 つを選ぶ
for group4 in itertools.combinations(vars_lists, 4):
    remaining = [v for v in vars_lists if v not in group4]
    # 残り 2 つから 1 つを選ぶ
        # 残った最後の 1 つを取得
    #group1 = next(v for v in group4 if v not in remaining)
    # リストとして格納：[4つリスト, 1つ目, 2つ目]
    splits2.append([
        list(group4),    # 4つのリストをリスト化
        remaining,       # 1つ目のリスト
        #group2           # 2つ目のリスト
    ])

print(f"全パターン数: {len(splits2)}")  # → 30

# 確認例
for i, pat in enumerate(splits2[:5], 1):
    four, one_a = pat
    print(f"{i:2d}:")
    print("  4つ→", four)
    print("  1つA→", one_a)
    #print("  1つB→", one_b)

全パターン数: 5
 1:
  4つ→ [[45, 49, 41, 37, 35, 25, 28, 53, 1, 54, 13, 20], [12, 47, 27, 33, 23, 31, 22, 50, 43, 46, 56, 17], [38, 52, 7, 57, 39, 4, 34, 6, 3, 32, 44, 58], [8, 10, 26, 0, 40, 15, 48, 24, 55, 11, 42, 5]]
  1つA→ [[2, 21, 29, 30, 51, 14, 16, 9, 36, 18, 19]]
 2:
  4つ→ [[45, 49, 41, 37, 35, 25, 28, 53, 1, 54, 13, 20], [12, 47, 27, 33, 23, 31, 22, 50, 43, 46, 56, 17], [38, 52, 7, 57, 39, 4, 34, 6, 3, 32, 44, 58], [2, 21, 29, 30, 51, 14, 16, 9, 36, 18, 19]]
  1つA→ [[8, 10, 26, 0, 40, 15, 48, 24, 55, 11, 42, 5]]
 3:
  4つ→ [[45, 49, 41, 37, 35, 25, 28, 53, 1, 54, 13, 20], [12, 47, 27, 33, 23, 31, 22, 50, 43, 46, 56, 17], [8, 10, 26, 0, 40, 15, 48, 24, 55, 11, 42, 5], [2, 21, 29, 30, 51, 14, 16, 9, 36, 18, 19]]
  1つA→ [[38, 52, 7, 57, 39, 4, 34, 6, 3, 32, 44, 58]]
 4:
  4つ→ [[45, 49, 41, 37, 35, 25, 28, 53, 1, 54, 13, 20], [38, 52, 7, 57, 39, 4, 34, 6, 3, 32, 44, 58], [8, 10, 26, 0, 40, 15, 48, 24, 55, 11, 42, 5], [2, 21, 29, 30, 51, 14, 16, 9, 36, 18, 19]]
  1つA→ [[12, 47, 27, 33, 23,

In [39]:
import random


mouse = 2
random.seed(30)

# 0～35 のリストを作成してシャッフル
numbers = list(range(experiment_list[mouse].shape[0]))
random.shuffle(numbers)

# 6 個ずつスライスして変数に格納
dataset1 = numbers[0:9]
dataset2 = numbers[9:18]
dataset3 = numbers[18:26]
dataset4 = numbers[26:34]
dataset5 = numbers[34:42]

In [40]:
import itertools


vars_lists = [dataset1, dataset2, dataset3, dataset4, dataset5]

splits3 = []
# 6 つから 4 つを選ぶ
for group4 in itertools.combinations(vars_lists, 4):
    remaining = [v for v in vars_lists if v not in group4]
    # 残り 2 つから 1 つを選ぶ
        # 残った最後の 1 つを取得
    #group1 = next(v for v in group4 if v not in remaining)
    # リストとして格納：[4つリスト, 1つ目, 2つ目]
    splits3.append([
        list(group4),    # 4つのリストをリスト化
        remaining,       # 1つ目のリスト
        #group2           # 2つ目のリスト
    ])

print(f"全パターン数: {len(splits3)}")  # → 30

# 確認例
for i, pat in enumerate(splits3[:5], 1):
    four, one_a = pat
    print(f"{i:2d}:")
    print("  4つ→", four)
    print("  1つA→", one_a)
    #print("  1つB→", one_b)

全パターン数: 5
 1:
  4つ→ [[40, 37, 32, 30, 15, 27, 33, 23, 38], [6, 4, 10, 9, 22, 35, 26, 21, 41], [17, 11, 20, 12, 28, 19, 31, 2], [29, 7, 36, 0, 14, 5, 8, 24]]
  1つA→ [[25, 3, 16, 13, 1, 39, 18, 34]]
 2:
  4つ→ [[40, 37, 32, 30, 15, 27, 33, 23, 38], [6, 4, 10, 9, 22, 35, 26, 21, 41], [17, 11, 20, 12, 28, 19, 31, 2], [25, 3, 16, 13, 1, 39, 18, 34]]
  1つA→ [[29, 7, 36, 0, 14, 5, 8, 24]]
 3:
  4つ→ [[40, 37, 32, 30, 15, 27, 33, 23, 38], [6, 4, 10, 9, 22, 35, 26, 21, 41], [29, 7, 36, 0, 14, 5, 8, 24], [25, 3, 16, 13, 1, 39, 18, 34]]
  1つA→ [[17, 11, 20, 12, 28, 19, 31, 2]]
 4:
  4つ→ [[40, 37, 32, 30, 15, 27, 33, 23, 38], [17, 11, 20, 12, 28, 19, 31, 2], [29, 7, 36, 0, 14, 5, 8, 24], [25, 3, 16, 13, 1, 39, 18, 34]]
  1つA→ [[6, 4, 10, 9, 22, 35, 26, 21, 41]]
 5:
  4つ→ [[6, 4, 10, 9, 22, 35, 26, 21, 41], [17, 11, 20, 12, 28, 19, 31, 2], [29, 7, 36, 0, 14, 5, 8, 24], [25, 3, 16, 13, 1, 39, 18, 34]]
  1つA→ [[40, 37, 32, 30, 15, 27, 33, 23, 38]]


In [41]:
splits_list = [splits1, splits2, splits3]

In [ ]:
#mean_array2, std_array2 = [], []
#min_array2, max_array2 = [], []
#for mouse in tqdm.tqdm(range(len(mouse_list))):
#    mean_array2.append(mean_array[mouse][experiment_list[mouse]])
#    std_array2.append(std_array[mouse][experiment_list[mouse]])
#    min_array2.append(min_array[mouse][experiment_list[mouse]])
#    max_array2.append(max_array[mouse][experiment_list[mouse]])

100%|██████████| 3/3 [00:00<?, ?it/s]


In [ ]:
#np.save(f"{dl_folder}/model_{dl_number}/{date}_{dl_number}_means.npy", np.array(mean_array2))
#np.save(f"{dl_folder}/model_{dl_number}/{date}_{dl_number}_stds.npy", np.array(std_array2))
#np.save(f"{dl_folder}/model_{dl_number}/{date}_{dl_number}_mins.npy", np.array(min_array2))
#np.save(f"{dl_folder}/model_{dl_number}/{date}_{dl_number}_maxs.npy", np.array(max_array2))

In [42]:
np.save(f"{dl_folder}/model_{dl_number}/{date}_{dl_number}_selected_experiments_mouse1.npy", np.array(experiment_list[0]))
np.save(f"{dl_folder}/model_{dl_number}/{date}_{dl_number}_selected_experiments_mouse2.npy", np.array(experiment_list[1]))
np.save(f"{dl_folder}/model_{dl_number}/{date}_{dl_number}_selected_experiments_mouse3.npy", np.array(experiment_list[2]))

In [43]:
targetframe = 0 # n frame later label you rely on

def mini_max_signal_a(dataset, signal_min, signal_max):
    dataset_nor = []
    for i in range(len(dataset)):
        min_value = signal_min[i]
        max_value = signal_max[i]
        dataset_ex = []
        for t in range(len(dataset[i])-targetframe):
            if (max_value - min_value)>0:
                dataset_ex.append((dataset[i][t] - min_value) / (max_value - min_value))
            if (max_value - min_value)==0:
                dataset_ex.append(0)

        dataset_nor.append(dataset_ex)
    
    return dataset_nor

def mini_max_signal_b(dataset, signal_min, signal_max):
    dataset_nor = []
    for i in range(len(dataset)):
        min_value = signal_min[i]
        max_value = signal_max[i]
        dataset_ex = []
        for t in range(len(dataset[i])-abs(targetframe)):
            if (max_value - min_value)>0:
                dataset_ex.append((dataset[i][t+abs(targetframe)] - min_value) / (max_value - min_value))
            if (max_value - min_value)==0:
                dataset_ex.append(0)

        dataset_nor.append(dataset_ex)
    
    return dataset_nor

def z_score_signal(dataset, signal_ave, signal_std):
    dataset_nor = []
    for i in range(len(dataset)):
        ave_value = signal_ave[i]
        std_value = signal_std[i]
        dataset_ex = []
        for t in range(len(dataset[i])-targetframe):
            dataset_ex.append((dataset[i][t] - ave_value) / std_value)
        dataset_nor.append(dataset_ex)
    
    return dataset_nor



In [44]:
import numpy

# Create input data [-n,n+1]
def create_dataset(dataset, look_frame):
    dataX, dataY = [], []
    #for i in range(look_frame,len(dataset)-(look_frame)):
    for i in range(look_frame, len(dataset)):
        xset = []
        for j in range(dataset.shape[1]-1):
            #a = dataset[(i-look_frame):(i+look_frame+1), j]
            a = dataset[(i-look_frame):(i+1), j]
            xset.append(a)
        dataY.append(dataset[i, -1])      
        dataX.append(xset)
    return numpy.array(dataX), numpy.array(dataY)

In [45]:
def Data_preprocess(signal, pupil):
#def Data_preprocess(signal, pupil, signal_ave, signal_std):
    if targetframe >= 0:
        #signal = z_score_signal(signal, signal_ave, signal_std)
        signal = numpy.array(signal).transpose()
        pupil = [[x] for x in numpy.array(pupil)]
        moved_pupil = pupil.copy()
        del moved_pupil[:targetframe]
    else:
        #signal = z_score_signal(signal, signal_ave, signal_std)
        signal = numpy.array(signal).transpose()
        pupil = [[x] for x in numpy.array(pupil)]
        moved_pupil = pupil.copy()
        del moved_pupil[targetframe:]
    pupil =  numpy.array(moved_pupil)
    #pupil =  (pupil - pupil_min) / (pupil_max - pupil_min)
    #pupil =  (pupil - numpy.mean(pupil)) / numpy.std(pupil)
    #pupil =  (pupil - min(pupil)) / (max(pupil) - min(pupil))
    # Label proportion
    #print('The positive proportion is {:.2f}'.format(pupil.sum() / len(pupil)))
    # Create dataset
    dataset = numpy.hstack([signal.astype('float32'), pupil.astype('float32')])
    X, Y = create_dataset(dataset, look_frame)

    positive_rate = pupil.sum() / len(pupil)

    return X, Y, positive_rate

In [46]:
def multi_mouse(mouse, signal, Pupil_data, model):
    splits = splits_list[mouse]
    signal = signal[mouse]
    Pupil_data = Pupil_data[mouse]

    flat_list = [x for sublist in splits[model][0] for x in sublist]
    train_numbers = sorted(flat_list)
    valid_numbers = splits[model][1][0]
    #test_numbers  = splits[n][2]

    np.save(f"{save_dir}/{date}_{dl_number}_train_numbers_mouse{mouse}.npy", np.array(train_numbers))
    np.save(f"{save_dir}/{date}_{dl_number}_valid_numbers_mouse{mouse}.npy", np.array(valid_numbers))
    #np.save(f"{save_dir}/{date}_{dl_number}_test_numbers_mouse{mouse}.npy", np.array(test_numbers))

    TRratio_Ca = [signal[i] for i in train_numbers]
    VAratio_Ca = [signal[i] for i in valid_numbers]
    #TEratio_Ca = [signal[i] for i in test_numbers]
    TRPupil_data = [Pupil_data[i] for i in train_numbers]
    VAPupil_data = [Pupil_data[i] for i in valid_numbers]
    #TEPupil_data = [Pupil_data[i] for i in test_numbers]

    TR_signal_array = np.array(TRratio_Ca).transpose(1, 0, 2)
    TR_signal_array = TR_signal_array.reshape(TR_signal_array.shape[0], TR_signal_array.shape[1]*TR_signal_array.shape[2])

    # Training dataset
    trainX = numpy.empty([0, signal.shape[1], look_frame+1], dtype=numpy.float32)
    trainY = numpy.empty(0, dtype=numpy.float32)
    pr_tr = []
    for i in list(range(len(train_numbers))):
        X, Y, pr = Data_preprocess(TRratio_Ca[i], TRPupil_data[i])
        trainX = numpy.concatenate([trainX, X], axis=0)
        trainY = numpy.concatenate([trainY, Y], axis=0)
        pr_tr.append(pr)

    # Training dataset
    validX = numpy.empty([0, signal.shape[1], look_frame+1], dtype=numpy.float32)
    validY = numpy.empty(0, dtype=numpy.float32)
    pr_va = []
    for i in list(range(len(valid_numbers))):
        X, Y, pr = Data_preprocess(VAratio_Ca[i], VAPupil_data[i])
        validX = numpy.concatenate([validX, X], axis=0)
        validY = numpy.concatenate([validY, Y], axis=0)
        pr_va.append(pr)

    # Training dataset
    #testX = numpy.empty([0, signal.shape[1], look_frame+1], dtype=numpy.float32)
    #testY = numpy.empty(0, dtype=numpy.float32)
    #pr_te = []
    #for i in list(range(len(test_numbers))):
    #    X, Y, pr = Data_preprocess(TEratio_Ca[i], TEPupil_data[i])
    #    testX = numpy.concatenate([testX, X], axis=0)
    #    testY = numpy.concatenate([testY, Y], axis=0)
    #    pr_te.append(pr)

    # Transpose input
    input_train = trainX.transpose(0,2,1)
    input_valid = validX.transpose(0,2,1)
    #input_test = testX.transpose(0,2,1)

    posi_rate = np.array([sum(pr_tr)/len(pr_tr), sum(pr_va)/len(pr_va)])


    return input_train, input_valid, trainY, validY, TR_signal_array, posi_rate

In [47]:
import tqdm
import numpy

no_mouse = 3
ex_frames = 50


for n in tqdm.tqdm(range(len(splits1))):
    save_dir = os.path.join(os.path.join(dl_folder, f"model_{dl_number}"), f"model_{n}")
    os.makedirs(save_dir, exist_ok=True)


    #input_train, input_valid, input_test = [], [], []
    for mouse in range(no_mouse):
        #temp_train, temp_valid, temp_test, temp_trY, temp_vaY, temp_teY = multi_mouse(mouse, signal, Pupil_data)
        temp_train, temp_valid, temp_trY, temp_vaY, temp_tr_sig, temp_posi = multi_mouse(mouse, signal, Pupil_data, model=n)
        print(temp_posi)
        if mouse == 0:
            #input_train, input_valid, input_test, trainY, validY, testY = temp_train, temp_valid, temp_test, temp_trY, temp_vaY, temp_teY
            input_train, input_valid, trainY, validY, TR_signal_array  = temp_train, temp_valid, temp_trY, temp_vaY, temp_tr_sig
        else:
            input_train = np.concatenate((input_train, temp_train), axis=0)
            input_valid = np.concatenate((input_valid, temp_valid), axis=0)
            #input_test = np.concatenate((input_test, temp_test), axis=0)
            trainY = np.concatenate((trainY, temp_trY), axis=0)
            validY = np.concatenate((validY, temp_vaY), axis=0)
            #testY = np.concatenate((testY, temp_teY), axis=0)
            TR_signal_array = np.concatenate((TR_signal_array, temp_tr_sig), axis=1)



    signal_min = []
    signal_max = []
    signal_ave = []
    signal_std = []
    for area in range(TR_signal_array.shape[0]):
        signal_min.append(min(TR_signal_array[area]))
        signal_max.append(max(TR_signal_array[area]))
        signal_ave.append(np.mean(TR_signal_array[area]))
        signal_std.append(np.std(TR_signal_array[area]))
    signal_min = np.array(signal_min)
    signal_max = np.array(signal_max)
    signal_ave = np.array(signal_ave)
    signal_std = np.array(signal_std)

    #for dataset in range(input_train.shape[0]):
    #    for area in range(input_train.shape[2]):
    #        input_train[dataset, :, area] = (input_train[dataset, :, area] - signal_ave[area]) / signal_std[area]

    #for dataset in range(input_valid.shape[0]):
    #    for area in range(input_valid.shape[2]):
    #        input_valid[dataset, :, area] = (input_valid[dataset, :, area] - signal_ave[area]) / signal_std[area]   

    #for dataset in range(input_test.shape[0]):
    #    for area in range(input_test.shape[2]):
    #        input_test[dataset, :, area] = (input_test[dataset, :, area] - signal_ave[area]) / signal_std[area]


    #for dataset in range(input_train.shape[0]):
    #    for area in range(input_train.shape[2]):
    #        input_train[dataset, :, area] = (input_train[dataset, :, area] - signal_min[area]) / (signal_max[area] - signal_min[area])

    #for dataset in range(input_valid.shape[0]):
    #    for area in range(input_valid.shape[2]):
    #        input_valid[dataset, :, area] = (input_valid[dataset, :, area] - signal_min[area]) / (signal_max[area] - signal_min[area])

    #for dataset in range(input_test.shape[0]):
    #    for area in range(input_test.shape[2]):
    #        input_test[dataset, :, area] = (input_test[dataset, :, area] - signal_min[area]) / (signal_max[area] - signal_min[area])

    np.save(f"{save_dir}/{date}_{dl_number}_train_signal_min.npy", signal_min)
    np.save(f"{save_dir}/{date}_{dl_number}_train_signal_max.npy", signal_max)
    np.save(f"{save_dir}/{date}_{dl_number}_train_signal_ave.npy", signal_ave)
    np.save(f"{save_dir}/{date}_{dl_number}_train_signal_std.npy", signal_std)

    # shuffle train y labels
    #ex_no = int(input_train.shape[0]/ex_frames)
    #for ex in range(ex_no):
    #    copy_array = trainY[ex*ex_frames:(ex+1)*ex_frames].copy()
    #    np.random.shuffle(copy_array)
    #    trainY[ex*ex_frames:(ex+1)*ex_frames] = copy_array

    np.save(f"{save_dir}/{date}_{dl_number}_train_features.npy", input_train)
    np.save(f"{save_dir}/{date}_{dl_number}_train_targets.npy", trainY)
    np.save(f"{save_dir}/{date}_{dl_number}_valid_features.npy", input_valid)
    np.save(f"{save_dir}/{date}_{dl_number}_valid_targets.npy", validY)
    #np.save(f"{save_dir}/{date}_{dl_number}_test_features.npy", input_test)
    #np.save(f"{save_dir}/{date}_{dl_number}_test_targets.npy", testY)

  0%|          | 0/5 [00:00<?, ?it/s]

[0.42463235 0.4046875 ]
[0.41640625 0.45      ]
[0.43897059 0.375     ]


 20%|██        | 1/5 [00:00<00:02,  1.51it/s]

[0.41875   0.4296875]
[0.41861702 0.43854167]
[0.42794118 0.421875  ]


 40%|████      | 2/5 [00:01<00:01,  1.54it/s]

...

 60%|██████    | 3/5 [00:01<00:01,  1.60it/s]

[0.41818182 0.43055556]
[0.42526596 0.4125    ]
[0.4155303  0.46805556]


 80%|████████  | 4/5 [00:02<00:00,  1.63it/s]

[0.42272727 0.41388889]
[0.42420213 0.41666667]
[0.42689394 0.42638889]


100%|██████████| 5/5 [00:03<00:00,  1.50it/s]
